# 01 · EDA: structure, signal, leakage, drift

Answer in this order (write the answers into docs/PLAYBOOK.md §Event log, not only here):
1. What is one row? Which columns are ids, dates, entities, categoricals, feature blocks?
2. How were train/test split (dates interleaved / later / new entities)? → CV choice.
3. Target balance / distribution and the metric's naive baseline (always-up, mean, ...).
4. Which features carry per-date signal (IC t-stat)?
5. Leaks and drift: id order, recoverable date order, train-vs-test separability.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
# os.environ["QRTC_CONFIG"] = "configs/rehearsal/stock_sign.toml"  # rehearsal; unset at the event
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60, "display.width", 160)
from qrtc import config, io, eda, features as F, cv, train, tracking, ensemble, postprocess, submit
from qrtc.metrics import get_metric
spec = config.load_spec()
spec

In [ ]:
ds = io.load_dataset(spec)
feats = io.feature_columns(ds.train, spec)
eda.overview(ds.train)

In [ ]:
eda.split_structure(ds.train, ds.test, spec)

## Target and naive baselines

In [ ]:
display(eda.target_summary(ds.y, spec.task))
if spec.task in ("binary", "multiclass"):
    y = ds.y_vector
    majority = np.bincount(y).argmax()
    print("always-majority accuracy:", (y == majority).mean())

## Feature blocks (RET_1..RET_20 style)

In [ ]:
prefixes = sorted({c.rsplit("_", 1)[0] for c in feats if c.rsplit("_", 1)[-1].isdigit()})
blocks = {p: F.col_block(ds.train, p) for p in prefixes}
{p: len(c) for p, c in blocks.items()}

## Signal scan: per-date IC (Spearman of each feature with the target, averaged over dates)

In [ ]:
y1 = ds.y_vector if ds.y_vector.ndim == 1 else ds.y_vector[:, 0]
dates = ds.train[spec.date_col].to_numpy() if spec.date_col else None
scan = eda.signal_scan(ds.train[feats], y1, dates, top=40)
scan

In [ ]:
col = "t_stat" if "t_stat" in scan else "spearman"
ax = scan[col].sort_values().plot.barh(figsize=(6, 9), title=f"feature signal ({col})")
ax.axvline(0, color="k", lw=0.8)
plt.tight_layout()

## Leakage checks (ASK THE ORGANISERS before exploiting anything found here)

In [ ]:
eda.leakage_checks(ds.train, y1, spec)

In [ ]:
# Recoverable date order? (RET_k of date t+1 == RET_{k-1} of date t for the same entity)
ret_block = next((blocks[p] for p in blocks if p.upper().startswith("RET")), None)
if ret_block and spec.group_col:
    links = eda.lag_chain_links(ds.train, spec.group_col, ret_block)
    print(len(links), "lag-chain links found")
    display(links.head())

## Drift: can a model tell train from test?

In [ ]:
auc, imp = eda.adversarial_validation(ds.train, ds.test, feats, seed=spec.seed)
print(f"adversarial AUC = {auc:.3f}  (0.5 = no drift)")
imp.head(15)